# qlib_alpha158_supervised_cadence_v1

Executable evidence for the next simplification step: replace the RL-first training surface with a qlib Alpha158-style supervised screen, keep the mined SOXL factors as optional additions, and reuse the regime-aware trade-cadence overlay.

This is a diagnostic retraining screen, not a model promotion and not full RL deletion.

## Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` production composer.
- **Direct comparison baseline**: `quantbench_regime_cadence_screen_v1` supervised cadence screen using repo SOXL factor packs without Alpha158.
- **Role**: diagnose.
- **Target window**: `test_2022`, plus `test_2020`, `test_2023_2024`, `latest_2025_2026`, and `ytd_2026` slices.
- **Accepted trade-off**: cleaner non-RL feature/model surface even if the first Alpha158 screen does not pass promotion gates.
- **Changed layer**: feature, model_selection, execution cadence.
- **External reference**: qlib `Alpha158DL` feature family; Kaggle/Jane Street lesson is validation discipline only: time-ordered local validation, no leakage, and slice diagnostics before any promotion.
- **Closest success case**: `quantbench_regime_cadence_screen_v1` proved dynamic trade frequency is measurable inside the supervised path.
- **Closest failure case**: `quantbench_paper_soxl_v2` and the cadence screen showed single supervised rows often fail the 20% MaxDD gate.
- **Why this is not a duplicate**: this run adds a new qlib Alpha158-style feature surface and compares cadence rows after retraining on Alpha158 + mined features.
- **Comparison-table row key**: `profile` in `supervised_candidate_screen.csv` and `cadence_profile` in `cadence_pair_metrics.csv`.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError("repo root not found")


ROOT = find_repo_root()
CARD = ROOT / "runtime" / "model_selection" / "card" / "qlib_alpha158_supervised_cadence_v1"
CARD.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import torch
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass

from src.soxl_quant.app.quantbench_screen import QuantBenchGate
from src.soxl_quant.app.quantbench_supervised import (
    build_quantbench_supervised_report,
    quantbench_risk_overlay_configs,
    quantbench_trade_cadence_configs,
)
from src.soxl_quant.data import QLIB_ALPHA158_COLUMNS, build_dataset, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

print(f"root={ROOT.relative_to(ROOT)}")
print(f"card={CARD.relative_to(ROOT)}")

root=.
card=runtime/model_selection/card/qlib_alpha158_supervised_cadence_v1


## Feature Surface

The primary new surface is `qlib_alpha158`, sourced from the installed qlib `Alpha158DL` schema and implemented as a repo feature pack. The additional packs are curated mined SOXL signals already present in the harness, so this run tests Alpha158 plus known candidate overlays rather than adding another RL-specific feature schema.

In [2]:
feature_packs = (
    "qlib_alpha158",
    "release_signals_v1",
    "vix_term",
    "macro_signals",
    "soxx_signals",
)

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(feature_packs=feature_packs, profile_name="qlib_alpha158_supervised_cadence_v1")
features = dataset.handler.feature_frame
close = dataset.handler.price_frame["price"]

if isinstance(features.index, pd.MultiIndex) and "instrument" in features.index.names:
    feature_index = pd.to_datetime(features.index.get_level_values("datetime"))
else:
    feature_index = pd.to_datetime(features.index)
model_test_end = str(pd.Series(feature_index).max().date())
try:
    qlib_path = str(QLIB_DATA_PATH.relative_to(ROOT))
except ValueError:
    qlib_path = str(QLIB_DATA_PATH)
alpha_columns = [column for column in features.columns if str(column).startswith("QLIB_ALPHA158_")]
DATA_CONTEXT = {
    "qlib_data_path": qlib_path,
    "feature_rows": int(len(features)),
    "feature_columns": int(features.shape[1]),
    "feature_start": str(pd.Series(feature_index).min().date()),
    "feature_end": model_test_end,
    "feature_packs": list(feature_packs),
    "alpha158_columns_expected": int(len(QLIB_ALPHA158_COLUMNS)),
    "alpha158_columns_present": int(len(alpha_columns)),
}
print(json.dumps(DATA_CONTEXT, indent=2, sort_keys=True))

[25536:MainThread](2026-06-17 22:13:00,638) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[25536:MainThread](2026-06-17 22:13:01,116) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[25536:MainThread](2026-06-17 22:13:01,116) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/quantbench-regime-cadence-v1/data/qlib/soxl')}


[25536:MainThread](2026-06-17 22:13:03,480) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[25536:MainThread](2026-06-17 22:13:03,481) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[25536:MainThread](2026-06-17 22:13:03,481) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{
  "alpha158_columns_expected": 158,
  "alpha158_columns_present": 158,
  "feature_columns": 238,
  "feature_end": "2026-06-15",
  "feature_packs": [
    "qlib_alpha158",
    "release_signals_v1",
    "vix_term",
    "macro_signals",
    "soxx_signals"
  ],
  "feature_rows": 3838,
  "feature_start": "2011-03-11",
  "qlib_data_path": "data/qlib/soxl"
}


## Supervised Retraining Screen

The model set is deliberately compact: linear ridge, histogram gradient boosting, lagged ridge sequence features, and the lightweight TCN regressor already available in the QuantBench supervised registry. This keeps the loop non-RL while touching tabular, tree, lagged-sequence, and neural temporal candidates.

In [3]:
model_names = (
    "ridge_mse",
    "hist_gradient_mse",
    "sequence_lag_ridge_mse",
    "tcn_mse",
)

windows = {
    "test_2020": ("2020-01-01", "2020-12-31"),
    "test_2022": ("2022-01-01", "2022-12-31"),
    "test_2023_2024": ("2023-01-01", "2024-12-31"),
    "latest_2025_2026": ("2025-01-01", model_test_end),
    "ytd_2026": ("2026-01-01", model_test_end),
}

risk_configs = tuple(quantbench_risk_overlay_configs())
cadence_configs = tuple(quantbench_trade_cadence_configs())

report = build_quantbench_supervised_report(
    features,
    close,
    card_dir=CARD,
    root=ROOT,
    horizon=20,
    model_names=model_names,
    feature_packs=feature_packs,
    train_start="2010-03-11",
    train_end="2018-12-31",
    test_start="2020-01-01",
    test_end=model_test_end,
    profile_prefix="qlib_alpha158_supervised_cadence_v1",
    gate=QuantBenchGate(max_drawdown_pct=20.0),
    min_position=0.10,
    max_position=0.90,
    cost=0.0008,
    risk_configs=risk_configs,
    cadence_configs=cadence_configs,
    windows=windows,
    rolling_enabled=False,
)

print(json.dumps({
    "records": report["screen"]["records"],
    "eligible_count": report["screen"]["eligible_count"],
    "model_names": report["data"]["model_names"],
    "feature_columns": report["data"]["feature_count"],
}, indent=2, sort_keys=True))

{
  "eligible_count": 0,
  "feature_columns": 238,
  "model_names": [
    "ridge_mse",
    "hist_gradient_mse",
    "sequence_lag_ridge_mse",
    "tcn_mse"
  ],
  "records": 40
}


## Matched Cadence Deltas

The cadence overlay is evaluated against each matched no-cadence row. Negative trade and turnover deltas show lower trading frequency after regime throttling.

In [4]:
payloads_path = CARD / "supervised_payloads.json"
payloads = json.loads(payloads_path.read_text(encoding="utf-8"))
payload_by_profile = {payload["profile"]: payload for payload in payloads}
cadence_name = cadence_configs[0].name


def metric_at(payload: dict[str, Any], slice_name: str | None, metric: str) -> float:
    source = payload.get("test_proxy", {}) if slice_name is None else payload.get("slices", {}).get(slice_name, {})
    value = source.get(metric)
    return float(value) if value is not None and value == value else float("nan")


def safe_delta(left: float, right: float) -> float:
    if not np.isfinite(left) or not np.isfinite(right):
        return float("nan")
    return float(left - right)

rows: list[dict[str, Any]] = []
for cadence_payload in payloads:
    profile = str(cadence_payload["profile"])
    if f"_{cadence_name}" not in profile:
        continue
    base_profile = profile.replace(f"_{cadence_name}", "")
    base_payload = payload_by_profile.get(base_profile)
    if base_payload is None:
        continue
    row: dict[str, Any] = {
        "cadence_profile": profile,
        "base_profile": base_profile,
        "model_name": cadence_payload.get("model_name"),
        "paper_family": cadence_payload.get("paper_family"),
        "risk_overlay": (cadence_payload.get("risk_overlay") or {}).get("name", "none"),
        "trade_cadence": cadence_name,
    }
    for label, slice_name in (("proxy", None), ("test_2022", "test_2022"), ("latest_2025_2026", "latest_2025_2026")):
        for metric in ("return", "max_dd", "turnover", "trades", "cadence_blocked_days"):
            cadence_value = metric_at(cadence_payload, slice_name, metric)
            base_value = metric_at(base_payload, slice_name, metric)
            row[f"cadence_{label}_{metric}"] = cadence_value
            row[f"base_{label}_{metric}"] = base_value
            row[f"delta_{label}_{metric}"] = safe_delta(cadence_value, base_value)
    rows.append(row)

pair_frame = pd.DataFrame(rows)
if not pair_frame.empty:
    pair_frame["reduced_test_2022_trades"] = pair_frame["delta_test_2022_trades"] < 0
    pair_frame["reduced_test_2022_turnover"] = pair_frame["delta_test_2022_turnover"] < 0
    pair_frame = pair_frame.sort_values(
        ["reduced_test_2022_trades", "delta_test_2022_trades", "delta_test_2022_turnover", "delta_test_2022_return"],
        ascending=[False, True, True, False],
    ).reset_index(drop=True)

pair_path = CARD / "cadence_pair_metrics.csv"
pair_frame.to_csv(pair_path, index=False)
print(f"wrote {pair_path.relative_to(ROOT)} rows={len(pair_frame)}")
display(pair_frame.head(10))

wrote runtime/model_selection/card/qlib_alpha158_supervised_cadence_v1/cadence_pair_metrics.csv rows=20


,cadence_profile,base_profile,model_name,paper_family,risk_overlay,trade_cadence,cadence_proxy_return,base_proxy_return,delta_proxy_return,cadence_proxy_max_dd,...,base_latest_2025_2026_turnover,delta_latest_2025_2026_turnover,cadence_latest_2025_2026_trades,base_latest_2025_2026_trades,delta_latest_2025_2026_trades,cadence_latest_2025_2026_cadence_blocked_days,base_latest_2025_2026_cadence_blocked_days,delta_latest_2025_2026_cadence_blocked_days,reduced_test_2022_trades,reduced_test_2022_turnover
0,qlib_alpha158_supervised_cadence_v1_sequence_l...,qlib_alpha158_supervised_cadence_v1_sequence_l...,sequence_lag_ridge_mse,sequence_temporal,none,regime_cadence_fast_bull_slow_stress_v1,378.122688,326.807128,51.315560,67.306852,...,15.603207,-5.793586,108.0,343.0,-235.0,255.0,NaN,NaN,True,True
1,qlib_alpha158_supervised_cadence_v1_ridge_mse_...,qlib_alpha158_supervised_cadence_v1_ridge_mse,ridge_mse,linear_regression,none,regime_cadence_fast_bull_slow_stress_v1,553.072730,500.237269,52.835460,62.889068,...,14.959475,-5.002915,104.0,343.0,-239.0,259.0,NaN,NaN,True,True
2,qlib_alpha158_supervised_cadence_v1_tcn_mse_re...,qlib_alpha158_supervised_cadence_v1_tcn_mse,tcn_mse,sequence_convolutional,none,regime_cadence_fast_bull_slow_stress_v1,588.734175,1125.546852,-536.812677,65.543582,...,31.484257,-13.609329,127.0,343.0,-216.0,236.0,NaN,NaN,True,True
3,qlib_alpha158_supervised_cadence_v1_hist_gradi...,qlib_alpha158_supervised_cadence_v1_hist_gradi...,hist_gradient_mse,boosted_tree,none,regime_cadence_fast_bull_slow_stress_v1,686.090406,673.527499,12.562906,63.689102,...,25.375802,-9.819242,128.0,343.0,-215.0,218.0,NaN,NaN,True,True
4,qlib_alpha158_supervised_cadence_v1_ridge_mse_...,qlib_alpha158_supervised_cadence_v1_ridge_mse_...,ridge_mse,linear_regression,macro_budget_defensive_v1,regime_cadence_fast_bull_slow_stress_v1,33.039650,23.272491,9.767159,21.326640,...,9.016793,-1.167872,44.0,97.0,-53.0,221.0,NaN,NaN,True,True
5,qlib_alpha158_supervised_cadence_v1_sequence_l...,qlib_alpha158_supervised_cadence_v1_sequence_l...,sequence_lag_ridge_mse,sequence_temporal,macro_budget_defensive_v1,regime_cadence_fast_bull_slow_stress_v1,87.577060,80.984924,6.592136,25.701222,...,9.536093,-2.018717,43.0,98.0,-55.0,222.0,NaN,NaN,True,True
6,qlib_alpha158_supervised_cadence_v1_ridge_mse_...,qlib_alpha158_supervised_cadence_v1_ridge_mse_...,ridge_mse,linear_regression,macro_budget_bull_guard_v1,regime_cadence_fast_bull_slow_stress_v1,23.018657,19.359508,3.659149,23.530240,...,3.221341,-0.040000,14.0,17.0,-3.0,180.0,NaN,NaN,True,True
7,qlib_alpha158_supervised_cadence_v1_sequence_l...,qlib_alpha158_supervised_cadence_v1_sequence_l...,sequence_lag_ridge_mse,sequence_temporal,macro_budget_bull_guard_v1,regime_cadence_fast_bull_slow_stress_v1,-9.694334,-4.518868,-5.175465,30.913632,...,3.300641,-0.040000,14.0,17.0,-3.0,180.0,NaN,NaN,True,True
8,qlib_alpha158_supervised_cadence_v1_ridge_mse_...,qlib_alpha158_supervised_cadence_v1_ridge_mse_...,ridge_mse,linear_regression,macro_budget_lowdd_v1,regime_cadence_fast_bull_slow_stress_v1,17.372437,9.649516,7.722922,15.045640,...,2.135306,-0.426093,13.0,26.0,-13.0,293.0,NaN,NaN,True,True
9,qlib_alpha158_supervised_cadence_v1_ridge_mse_...,qlib_alpha158_supervised_cadence_v1_ridge_mse_...,ridge_mse,linear_regression,macro_budget_reentry_v1,regime_cadence_fast_bull_slow_stress_v1,0.504797,-5.513348,6.018145,29.927722,...,15.399155,-1.948746,75.0,108.0,-33.0,95.0,NaN,NaN,True,True


## Summary

This summary records whether Alpha158 + mined features are now runnable in the simplified non-RL path. Promotion remains disabled unless the existing gates pass with stronger validation evidence.

In [5]:
screen_path = CARD / "supervised_candidate_screen.csv"
screen_frame = pd.read_csv(screen_path)

if pair_frame.empty:
    best_pair: dict[str, Any] | None = None
else:
    best_pair = pair_frame.iloc[0].replace({np.nan: None}).to_dict()

eligible_profiles = screen_frame.loc[screen_frame["eligible"].astype(bool), "profile"].tolist() if not screen_frame.empty else []
turnover_reduction_rows = int(pair_frame.get("reduced_test_2022_turnover", pd.Series(dtype=bool)).sum())
trade_reduction_rows = int(pair_frame.get("reduced_test_2022_trades", pd.Series(dtype=bool)).sum())

summary = {
    "experiment": "qlib_alpha158_supervised_cadence_v1",
    "decision": "alpha158_supervised_screen_recorded_no_promotion",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "quantbench_regime_cadence_screen_v1",
    "role": "diagnose",
    "changed_layer": ["feature", "model_selection", "execution"],
    "target_windows": windows,
    "accepted_tradeoff": "cleaner non-RL qlib Alpha158 surface before deleting RL training code",
    "external_reference": {
        "qlib_alpha158_source": "qlib.contrib.data.loader.Alpha158DL.get_feature_config",
        "kaggle_jane_street_lessons": [
            "time-ordered validation before promotion",
            "leakage avoidance across train/test boundaries",
            "local slice diagnostics over single leaderboard-like proxy metrics",
        ],
        "kaggle_urls": [
            "https://www.kaggle.com/competitions/jane-street-market-prediction",
            "https://www.kaggle.com/competitions/jane-street-real-time-market-data-forecasting",
        ],
    },
    "data": DATA_CONTEXT,
    "model_names": list(model_names),
    "risk_overlays": [config.name for config in risk_configs],
    "trade_cadences": [config.name for config in cadence_configs],
    "screen_records": int(report["screen"]["records"]),
    "eligible_profiles": eligible_profiles,
    "cadence_pair_rows": int(len(pair_frame)),
    "test_2022_trade_reduction_rows": trade_reduction_rows,
    "test_2022_turnover_reduction_rows": turnover_reduction_rows,
    "best_cadence_pair": best_pair,
    "sidecars": {
        "supervised_summary_json": str((CARD / "supervised_summary.json").relative_to(ROOT)),
        "supervised_candidate_screen_csv": str(screen_path.relative_to(ROOT)),
        "supervised_payloads_json": str(payloads_path.relative_to(ROOT)),
        "cadence_pair_metrics_csv": str(pair_path.relative_to(ROOT)),
    },
    "notes": [
        "This is a non-RL qlib DatasetH-backed supervised retraining screen.",
        "The feature surface combines qlib Alpha158-style OHLCV factors with curated mined SOXL packs.",
        "This run keeps RL deletion incomplete; it establishes a replacement supervised surface first.",
    ],
    "next_action": (
        "If this path remains runnable under focused regressions, next prune/deprecate active RL training entrypoints "
        "in favor of supervised model-selection and cadence-controlled execution."
    ),
}

summary_path = CARD / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True, allow_nan=True) + "\n", encoding="utf-8")
print(json.dumps({
    "summary": str(summary_path.relative_to(ROOT)),
    "eligible_profiles": len(eligible_profiles),
    "cadence_pair_rows": len(pair_frame),
    "test_2022_trade_reduction_rows": trade_reduction_rows,
    "test_2022_turnover_reduction_rows": turnover_reduction_rows,
}, indent=2, sort_keys=True))

{
  "cadence_pair_rows": 20,
  "eligible_profiles": 0,
  "summary": "runtime/model_selection/card/qlib_alpha158_supervised_cadence_v1/summary.json",
  "test_2022_trade_reduction_rows": 20,
  "test_2022_turnover_reduction_rows": 20
}


In [6]:
from IPython.display import Markdown, display

best = summary.get("best_cadence_pair") or {}
best_profile = best.get("cadence_profile", "none")
best_base = best.get("base_profile", "none")
best_trade_delta = best.get("delta_test_2022_trades")
best_turnover_delta = best.get("delta_test_2022_turnover")
best_return_delta = best.get("delta_test_2022_return")

conclusion = f"""
## Conclusion

`qlib_alpha158_supervised_cadence_v1` retrained `{len(model_names)}` non-RL supervised candidates on `{DATA_CONTEXT['alpha158_columns_present']}` Alpha158-style columns plus curated mined SOXL packs, then applied `{len(risk_configs)}` risk overlays and `{len(cadence_configs)}` regime-cadence overlay through `{model_test_end}`.

Key metrics: `{report['screen']['records']}` candidate rows, `{len(pair_frame)}` matched cadence pairs, `{trade_reduction_rows}` pairs reduced `test_2022` trades, `{turnover_reduction_rows}` pairs reduced `test_2022` turnover, and `{len(eligible_profiles)}` rows passed the existing gate.

Best diagnostic pair by cadence-reduction sort: `{best_profile}` versus `{best_base}` with `delta_test_2022_trades={best_trade_delta}`, `delta_test_2022_turnover={best_turnover_delta}`, and `delta_test_2022_return={best_return_delta}`.

Decision: record the Alpha158 supervised screen with no promotion. This advances the replacement non-RL training surface, but the project still has not deleted the existing RL stack.

Artifacts: `supervised_candidate_screen.csv`, `cadence_pair_metrics.csv`, `supervised_payloads.json`, `supervised_summary.json`, and `summary.json` in `{CARD.relative_to(ROOT)}`.
"""
display(Markdown(conclusion))


## Conclusion

`qlib_alpha158_supervised_cadence_v1` retrained `4` non-RL supervised candidates on `158` Alpha158-style columns plus curated mined SOXL packs, then applied `4` risk overlays and `1` regime-cadence overlay through `2026-06-15`.

Key metrics: `40` candidate rows, `20` matched cadence pairs, `20` pairs reduced `test_2022` trades, `20` pairs reduced `test_2022` turnover, and `0` rows passed the existing gate.

Best diagnostic pair by cadence-reduction sort: `qlib_alpha158_supervised_cadence_v1_sequence_lag_ridge_mse_regime_cadence_fast_bull_slow_stress_v1` versus `qlib_alpha158_supervised_cadence_v1_sequence_lag_ridge_mse` with `delta_test_2022_trades=-181.0`, `delta_test_2022_turnover=-6.945019920318725`, and `delta_test_2022_return=11.8637471987`.

Decision: record the Alpha158 supervised screen with no promotion. This advances the replacement non-RL training surface, but the project still has not deleted the existing RL stack.

Artifacts: `supervised_candidate_screen.csv`, `cadence_pair_metrics.csv`, `supervised_payloads.json`, `supervised_summary.json`, and `summary.json` in `runtime/model_selection/card/qlib_alpha158_supervised_cadence_v1`.
